# Etapa 3 — Modelagem

Entrada: `../data/processed/dataset.csv`, da Etapa 2.
Saída: um classificador treinado, avaliado e interpretado, e um ranking dos
melhores horários da janela.

Um detalhe que define esta etapa: **não existe label pronta.** O site
classifica as condições para pesca, não para surf — e pesca gosta de mar
calmo, quase o oposto. Definir o que é uma condição boa para surfar é a
primeira tarefa aqui, e é uma decisão sua.

In [8]:
from pathlib import Path

DIR_PROC = Path("..") / "data" / "processed"

# TODO: imports (pandas, sklearn, matplotlib, ...)

import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk

## 1. Carregar o dataset processado

Confira que veio o que você espera: número de linhas, tipos, nulos. Se a
Etapa 2 mudou, isso aparece aqui.

In [9]:
df = pd.read_csv(DIR_PROC / "dataset.csv", parse_dates=["datetime"])

print("formato:", df.shape)
print("nulos no total:", df.isna().sum().sum())
print("intervalo:", df["datetime"].min(), "até", df["datetime"].max())
df.info()
df.head()

formato: (168, 13)
nulos no total: 0
intervalo: 2026-10-11 00:00:00 até 2026-10-17 23:00:00
<class 'pandas.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              168 non-null    datetime64[us]
 1   hora_dia              168 non-null    int64         
 2   periodo               168 non-null    str           
 3   dia_semana            168 non-null    int64         
 4   fim_de_semana         168 non-null    bool          
 5   e_dia                 168 non-null    bool          
 6   coeficiente           168 non-null    int64         
 7   altura_mare_m         168 non-null    float64       
 8   sentido               168 non-null    str           
 9   altura_onda_m         168 non-null    float64       
 10  direcao_onda          168 non-null    str           
 11  velocidade_vento_kmh  168 non-null    float64       
 1

,datetime,hora_dia,periodo,dia_semana,fim_de_semana,e_dia,coeficiente,altura_mare_m,sentido,altura_onda_m,direcao_onda,velocidade_vento_kmh,direcao_vento
0,2026-10-11 00:00:00,0,madrugada,6,True,False,92,0.94,subindo,1.0,ESE,21.0,SE
1,2026-10-11 01:00:00,1,madrugada,6,True,False,92,1.28,subindo,1.0,ESE,22.0,SE
2,2026-10-11 02:00:00,2,madrugada,6,True,False,92,1.61,subindo,1.0,ESE,20.0,SE
3,2026-10-11 03:00:00,3,madrugada,6,True,False,92,1.94,subindo,1.0,ESE,17.0,ESE
4,2026-10-11 04:00:00,4,madrugada,6,True,False,92,2.27,subindo,1.0,ESE,21.0,ESE


**Conclusões da seção 1**

O `dataset.csv` carregou com 168 linhas e 13 colunas, sem nulos, cobrindo de 11/out 00:00 a 17/out 23:00 de 2026, como esperado da Etapa 2. Os tipos vieram como previstos: `datetime` como data; `fim_de_semana` e `e_dia` como booleanos; `hora_dia`, `dia_semana` e `coeficiente` como inteiros; `altura_mare_m`, `altura_onda_m` e `velocidade_vento_kmh` como `float64`; e `periodo`, `sentido`, `direcao_onda` e `direcao_vento` como texto (categorias).

## 2. Definir a label

Escreva, em código, a regra que separa condições ruins, boas e ótimas (ou
outro conjunto de classes, se preferir).

- Quais variáveis entram na regra: altura da onda, vento, maré, direção?
- Quais limiares? De onde vem o número — leitura, conversa com surfista,
  distribuição dos próprios dados? Registre a origem, mesmo que seja um chute
  informado.
- Como as classes ficaram distribuídas? Se 95% das horas caírem numa classe,
  a métrica do modelo vai mentir para você.

Documente a regra em markdown antes de seguir. Tudo depois disso depende dela.

**Regra da label**

A label classifica cada **hora de luz** (`e_dia` verdadeiro, 91 das 168 horas) em **ruim**, **boa** ou **ótima**. As horas de noite ficam fora do modelo, porque não são utilizáveis para surfar e fariam a classe "ruim" ser dominada por `e_dia`.

A regra usa só duas variáveis, **altura da onda** e **velocidade do vento**. Maré e direção do vento ficam de fora: a maré não mostrou relação com a onda na EDA, e a direção do vento praticamente não varia na janela (ENE, E, ESE e poucas horas de SE). As duas variáveis de fora entram como features na seção 3, e a importância que o modelo der a elas é um teste do que a regra deixou de captar.

Premissa: **onda maior e vento mais fraco são melhores**. Cada variável dá de 0 a 2 pontos, e a soma (0 a 4) define a classe:

| Variável | Pontos | Corte | Horas |
|---|---|---|---|
| Onda | 2 / 1 / 0 | ≥ 1,4 m / 1,2 a 1,3 m / ≤ 1,1 m | 13 / 64 / 14 |
| Vento | 2 / 1 / 0 | ≤ 20 / 21 a 23 / ≥ 24 km/h | 40 / 39 / 12 |

Soma de 0 a 2: **ruim**; 3: **boa**; 4: **ótima**.

**Origem dos cortes:** a distribuição dos próprios dados. Os da onda separam o valor máximo da janela e a cauda baixa. Os do vento são os terços da distribuição. **Não vêm de uma fonte externa**, então são um chute informado, e as classes são **relativas a esta semana**: "ótima" quer dizer "entre as melhores horas desta janela", e não que o surfe seja excelente.

**Distribuição das classes:** ruim 50 (55%), boa 32 (35%), ótima 9 (10%). A classe "ótima" é pequena, e todas as suas horas estão em 14 e 15/10, o que afeta a divisão treino/teste e torna a acurácia pouco informativa.

In [10]:
luz = df[df["e_dia"]].copy()
print(len(luz), "horas de luz\n")

print("Onda (m), número de horas por valor:")
print(luz["altura_onda_m"].value_counts().sort_index())

print("\nVento (km/h):")
print(luz["velocidade_vento_kmh"].describe())

print("\nOnda por dia (horas de luz):")
print(luz.groupby(luz["datetime"].dt.date)["altura_onda_m"].agg(["min", "mean", "max"]).round(2))

print("\nOnda x vento (vento em 3 faixas de mesmo tamanho):")
print(pd.crosstab(luz["altura_onda_m"], pd.qcut(luz["velocidade_vento_kmh"], 3, duplicates="drop")))

91 horas de luz

Onda (m), número de horas por valor:
altura_onda_m
0.9     1
1.0    12
1.1     1
1.2    35
1.3    29
1.4    13
Name: count, dtype: int64

Vento (km/h):
count    91.000000
mean     21.186813
std       2.699432
min      13.000000
25%      19.000000
50%      21.000000
75%      23.000000
max      28.000000
Name: velocidade_vento_kmh, dtype: float64

Onda por dia (horas de luz):
            min  mean  max
datetime                  
2026-10-11  0.9  0.99  1.0
2026-10-12  1.2  1.22  1.3
2026-10-13  1.2  1.23  1.3
2026-10-14  1.3  1.37  1.4
2026-10-15  1.3  1.33  1.4
2026-10-16  1.2  1.27  1.3
2026-10-17  1.1  1.19  1.2

Onda x vento (vento em 3 faixas de mesmo tamanho):
velocidade_vento_kmh  (12.999, 20.0]  (20.0, 23.0]  (23.0, 28.0]
altura_onda_m                                                   
0.9                                0             0             1
1.0                                2             4             6
1.1                                1             0 

In [11]:
def pontos_onda(h):
    if h >= 1.4:
        return 2
    if h >= 1.2:
        return 1
    return 0


def pontos_vento(v):
    if v <= 20:
        return 2
    if v <= 23:
        return 1
    return 0


luz["pontos_onda"] = luz["altura_onda_m"].apply(pontos_onda)
luz["pontos_vento"] = luz["velocidade_vento_kmh"].apply(pontos_vento)
luz["pontos"] = luz["pontos_onda"] + luz["pontos_vento"]
luz["label"] = luz["pontos"].map(
    lambda p: "ruim" if p <= 2 else ("boa" if p == 3 else "ótima")
)

print(luz["pontos"].value_counts().sort_index())
print()
print(luz["label"].value_counts())
print(luz["label"].value_counts(normalize=True).round(2))
print()
print(pd.crosstab(luz["datetime"].dt.date, luz["label"]))

pontos
0     7
1     9
2    34
3    32
4     9
Name: count, dtype: int64

label
ruim     50
boa      32
ótima     9
Name: count, dtype: int64
label
ruim     0.55
boa      0.35
ótima    0.10
Name: proportion, dtype: float64

label       boa  ruim  ótima
datetime                    
2026-10-11    0    13      0
2026-10-12    3    10      0
2026-10-13    3    10      0
2026-10-14    1     4      8
2026-10-15   12     0      1
2026-10-16    5     8      0
2026-10-17    8     5      0


## 3. Features e alvo

Separe `X` e `y`.

- Quais colunas entram como feature? Cuidado: usar diretamente as variáveis da
  sua regra faz o modelo apenas redescobrir o `if` que você escreveu. Isso é
  esperado aqui — mas você precisa saber que é isso que está acontecendo.
- Direção do vento e da onda são texto. Como transformar em número sem inventar
  uma ordem que não existe?

## 4. Treino e teste

Divida os dados. Vale estratificar pelas classes? A ordem temporal importa —
sortear linhas de um mesmo dia entre treino e teste vaza informação?

## 5. Treinar

Treine pelo menos dois modelos de famílias diferentes (por exemplo uma árvore
e um modelo baseado em distância) para ter com o que comparar.

## 6. Avaliar

- Acurácia é suficiente aqui, com as classes desbalanceadas como estão?
- Olhe a matriz de confusão: que erro o modelo comete?
- Um único split é confiável com poucas linhas, ou vale validação cruzada?

## 7. Interpretar

Um modelo que acerta e você não sabe por quê não serve para responder nada.

- Plote a árvore. Os cortes que ela encontrou parecem com os seus limiares?
- Quais features o modelo considera mais importantes? Bate com sua intuição?
- Se a acurácia veio altíssima, explique o motivo — não celebre.

## 8. Responder a pergunta

Aplique o modelo na janela inteira e produza a resposta para Felipe e Nicholas:

- Quais os melhores dias da janela de previsão?
- Em média, qual o melhor horário do dia?
- Em que grau essa resposta é uma previsão do mar, e em que grau é apenas a
  sua regra reescrita?

## 9. Conclusão e limites

Em texto:

1. A recomendação, em uma frase.
2. O teto deste modelo: ele reproduz a sua heurística, então é tão bom quanto
   ela. O que faria dela uma regra melhor?
3. O que a janela de ~7 dias e o volume de linhas impedem de afirmar.
4. Como você responderia "qual a melhor época do ano para surfar em João
   Pessoa" — que dado faltaria, e como coletá-lo.